# Mahalanobis Distance Matching
This script is an alternative to `match_cells.ipynb`. It matches treatment and control cells using Mahalanobis Distance Matching (MDM) rather than PSM. The script contains covariate balance diagnostics for direct comparison to PSM.

In [ ]:
# Select PA
site_id = 2017

In [ ]:
from pathlib import Path
import sys
import os
import ee
import numpy as np
import pandas as pd
import geemap

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    EE_CRS_METERS,
    PSM_CELL_SIZE,
    COVARIATES,
    BIOME_ASSET_ID,
    HGFC_ASSET_ID,
)

from absolute_effectiveness.site_selector import SiteSelector
from psm.prepare_pa_grid import load_pa_candidate_cells
from psm.covariates import build_resampled_covariates
from psm.cell_features import extract_cells_with_covariates
from psm.match_cells import (
    filter_matched_grids,
    save_matching_outputs,
)

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()

EE_CRS_1km = ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE)

## Data Prep
Load candidate cells and covariates.

In [ ]:
pa_ctx = load_pa_candidate_cells(site_id, site_selector)
PA_ID = pa_ctx["PA_ID"]
test_sites = pa_ctx["test_sites"]
site_geom = pa_ctx["site_geom"]
treatment_cells = pa_ctx["treatment_cells"]
control_cells = pa_ctx["control_cells"]
grid_fc = pa_ctx["grid_fc"]
covariates = build_resampled_covariates(EE_CRS_1km)

## Match Cells

In [ ]:
# Aggregate covariates within grid cells
grid_fc, cells_df = extract_cells_with_covariates(grid_fc, covariates, EE_CRS_1km)

In [ ]:
CALIPER_MAHALANOBIS = 3.0
N_NEIGHBORS = 4

# Compute scaler and covariance from CONTROL group only (Stuart 2010)
control_only = cells_df[cells_df["protected"] == 0]
scaler = StandardScaler()
X_control = scaler.fit_transform(control_only[COVARIATES].values)

cov_matrix = np.cov(X_control.T)
try:
    inv_cov = np.linalg.inv(cov_matrix)
except np.linalg.LinAlgError:
    print("Warning: covariance matrix is singular; using pseudo-inverse")
    inv_cov = np.linalg.pinv(cov_matrix)

# Apply the control-derived scaling to ALL cells (treatment + control)
X_pa = scaler.transform(cells_df[COVARIATES].values)
for i, col in enumerate(COVARIATES):
    cells_df[f"_scaled_{col}"] = X_pa[:, i]

# Split treatment and control cells
treat_df = cells_df[cells_df["protected"] == 1].copy().reset_index(drop=True)
control_df = cells_df[cells_df["protected"] == 0].copy().reset_index(drop=True)

print(f"Treatment cells: {len(treat_df)}")
print(f"Control cells: {len(control_df)}")

scaled_cols = [f"_scaled_{c}" for c in COVARIATES]
matches = []

# Nearest neighbor matching by Mahalanobis distance,
# exact match on country and ecoregion (with biome as a fallback)

for (country, ecoregion), treat_sub in treat_df.groupby(["country", "ecoregion"]):
    control_country = control_df[control_df["country"] == country]

    if len(control_country) == 0:
        print(f"  ({country}, ecoregion {ecoregion}): no controls in country, "
              f"skipping {len(treat_sub)} treatment cells")
        continue

    control_sub = control_country[control_country["ecoregion"] == ecoregion]

    if len(control_sub) == 0:
        biome = treat_sub["biome"].iloc[0]
        control_sub = control_country[control_country["biome"] == biome]
        fallback = "biome"
        print(f"  ({country}, ecoregion {ecoregion}): no within-ecoregion controls, "
              f"falling back to biome {biome} ({len(control_sub)} controls)")
    else:
        fallback = None

    if len(control_sub) == 0:
        print(f"  ({country}, ecoregion {ecoregion}): no controls at any fallback level, "
              f"skipping {len(treat_sub)} treatment cells")
        continue

    n_neighbors = min(N_NEIGHBORS, len(control_sub))
    nn = NearestNeighbors(
        n_neighbors=n_neighbors,
        metric="mahalanobis",
        metric_params={"VI": inv_cov},
    )
    nn.fit(control_sub[scaled_cols].values)

    distances, indices = nn.kneighbors(treat_sub[scaled_cols].values)

    for i, treat_row in enumerate(treat_sub.itertuples()):
        for rank, (dist, j) in enumerate(zip(distances[i], indices[i]), start=1):
            if dist <= CALIPER_MAHALANOBIS:
                control_row = control_sub.iloc[j]
                matches.append({
                    "treat_cell_id": treat_row.cell_ID,
                    "control_cell_id": control_row["cell_ID"],
                    "mahalanobis_distance": float(dist),
                    "match_rank": rank,
                    "match_country": country,
                    "match_ecoregion": ecoregion,
                    "match_fallback": fallback,
                })

match_df = pd.DataFrame(matches).sort_values("treat_cell_id").reset_index(drop=True)

# === Matching summary ===
print(f"\nResults:")
print(f"  Total matched pairs: {len(match_df)}")
print(f"  Unique treatment cells matched: {match_df['treat_cell_id'].nunique()}")
print(f"  Unique control cells used: {match_df['control_cell_id'].nunique()}")
unmatched_treat = set(treat_df["cell_ID"]) - set(match_df["treat_cell_id"])
print(f"  Treatment cells with no match: {len(unmatched_treat)}")
match_coverage = match_df["treat_cell_id"].nunique() / len(treat_df) if len(treat_df) > 0 else 0
print(f"  Match coverage: {match_coverage:.1%}")

if len(match_df) > 0:
    avg_matches = match_df.groupby("treat_cell_id").size().mean()
    print(f"  Avg matches per matched treatment cell: {avg_matches:.2f}")

    control_reuse = match_df.groupby("control_cell_id").size()
    print(f"  Control reuse: min={control_reuse.min()}, "
          f"max={control_reuse.max()}, "
          f"mean={control_reuse.mean():.1f}")

    print(f"  Mahalanobis distance: mean={match_df['mahalanobis_distance'].mean():.3f}, "
          f"max={match_df['mahalanobis_distance'].max():.3f}")

In [ ]:
# Save matched cells
matched_grids = filter_matched_grids(grid_fc, match_df)
# save_matching_outputs(matched_grids, match_df, PA_ID)

## Diagnostics

In [ ]:
# === Diagnostic 1: Pairwise covariate balance check ===
# Use the standardized differences test (Feng et al. 2022) to check that covariates
# are balanced between treatment and control cells after matching.
# This test verifies the validity of the PSM.

# Better metric of per-match covariate balance, but doesn't have a before/after comparison.

# Build matched treatment and control DataFrames
# Each row of match_df becomes one row in each: treat row has treatment covariates,
# control row has control covariates. Controls appear multiple times (reuse).
matched_treat = match_df.merge(
    cells_df[["cell_ID"] + COVARIATES],
    left_on="treat_cell_id",
    right_on="cell_ID",
).drop(columns="cell_ID")

matched_control = match_df.merge(
    cells_df[["cell_ID"] + COVARIATES],
    left_on="control_cell_id",
    right_on="cell_ID",
).drop(columns="cell_ID")

def compute_pair_smd(matched_t_vals, matched_c_vals, full_t_vals, full_c_vals):
    """
    Pair-level standardized mean difference.

    Returns the mean absolute pair distance (in pooled-SD units) plus
    the 90th percentile, which surfaces worst-pair imbalance.

    Pooled SD is computed on the FULL (unmatched) treatment and control pools
    to anchor the metric to the original covariate scale.

    Parameters
    ----------
    matched_t_vals, matched_c_vals : pandas Series, equal length
        Covariate values for matched treatment and control cells (in pair order).
    full_t_vals, full_c_vals : pandas Series
        Covariate values for the full unmatched treatment and control pools.
        Used to compute the pooled SD that anchors the metric.

    Returns
    -------
    (mean_abs_smd, p90_abs_smd, signed_smd) : tuple of floats
    """
    var_full_t = full_t_vals.var()
    var_full_c = full_c_vals.var()
    pooled_sd = np.sqrt((var_full_t + var_full_c) / 2)
    if pooled_sd == 0:
        return 0.0, 0.0, 0.0

    pair_diffs = (matched_t_vals.values - matched_c_vals.values) / pooled_sd
    return (
        np.abs(pair_diffs).mean(),
        np.percentile(np.abs(pair_diffs), 90),
        pair_diffs.mean(),  # signed mean for directional info
    )

print("Pair-level balance check")
print("Mean absolute pair SMD (lower = better individual match quality)")
print("Threshold: < 0.25 = acceptable; < 0.10 = excellent")
print("=" * 90)
print(f"{'covariate':<20s} {'mean |smd|':>12s} {'p90 |smd|':>12s} {'signed mean':>14s} {'verdict':>20s}")
print("-" * 90)

unmatched_treat = cells_df[cells_df["protected"] == 1]
unmatched_control = cells_df[cells_df["protected"] == 0]

for col in COVARIATES:
    mean_abs, p90, signed = compute_pair_smd(
        matched_treat[col],
        matched_control[col],
        unmatched_treat[col],
        unmatched_control[col],
    )

    if mean_abs < 0.10:
        verdict = "excellent"
    elif mean_abs < 0.25:
        verdict = "acceptable"
    elif mean_abs < 0.5:
        verdict = "IMBALANCED"
    else:
        verdict = "IMBALANCED"

    print(f"{col:<20s} {mean_abs:>12.3f} {p90:>12.3f} {signed:>+14.3f} {verdict:>20s}")

    # Collect results for saving
    results = []
    for col in COVARIATES:
        mean_abs, p90, signed = compute_pair_smd(
            matched_treat[col],
            matched_control[col],
            unmatched_treat[col],
            unmatched_control[col],
        )

        if mean_abs < 0.10:
            verdict = "excellent"
        elif mean_abs < 0.25:
            verdict = "acceptable"
        elif mean_abs < 0.5:
            verdict = "IMBALANCED"
        else:
            verdict = "IMBALANCED"

        results.append({
            "covariate": col,
            "mean_abs_smd": mean_abs,
            "p90_abs_smd": p90,
            "signed_mean": signed,
            "verdict": verdict
        })

    # Save as CSV
    results_df = pd.DataFrame(results)
    # results_df.to_csv(f'results/MDM_balance_{site_id}.csv', index=False)

## Visualization

In [ ]:
# Visualize covariates

Map = geemap.Map()
Map.add_basemap("CartoDB.DarkMatter")
Map.centerObject(grid_fc)

Map.addLayer(
    covariates.select("elevation"),
    {"min": 85, "max": 4200, "palette": ["blue", "green", "yellow", "red"]},
    "Elevation", 0
)
Map.addLayer(
    covariates.select("slope"),
    {"min": 0, "max": 24, "palette": ["blue", "green", "yellow", "red"]},
    "Slope", 0
)
Map.addLayer(
    covariates.select("treecover2000"),
    {"min": 0, "max": 100, "palette": ["white", "green"]},
    "Tree Cover (2000)", 0
)
Map.addLayer(
    covariates.select("travel_time"),
    {"min": 56, "max": 2731, "palette": ["blue", "green", "yellow", "red"]},
    "Travel Time (2015)", 0
)
Map.addLayer(
    covariates.select("log_pop_density"),
    {"min": 0, "max": 5, "palette": ["blue", "green", "yellow", "red"]},
    "Population Density (2000)", 0
)
Map.addLayer(
    covariates.select("human_footprint"),
    {"min": 0, "max": 44, "palette": ["blue", "green", "yellow", "red"]},
    "Human Footprint (1993)", 0
)
Map.addLayer(
    covariates.select("ag_suitability"),
    {"min": 200, "max": 9800, "palette": ["blue", "green", "yellow", "red"]},
    "Agricultural Suitability (2001-2020)", 0
)

Map

In [ ]:
# Visualize matched cells

Map = geemap.Map()
# Map.add_basemap("CartoDB.Positron")
Map.add_basemap("CartoDB.DarkMatter")
ecoRegions = ee.FeatureCollection(BIOME_ASSET_ID)

color_updates = [
    {"ECO_ID": 204, "COLOR": '#B3493B'},
    {"ECO_ID": 245, "COLOR": '#267400'},
    {"ECO_ID": 259, "COLOR": '#004600'},
    {"ECO_ID": 286, "COLOR": '#82F178'},
    {"ECO_ID": 316, "COLOR": '#E600AA'},
    {"ECO_ID": 453, "COLOR": '#5AA500'},
    {"ECO_ID": 317, "COLOR": '#FDA87F'},
    {"ECO_ID": 763, "COLOR": '#A93800'},
]

def add_style_property(feature):
    color = feature.get('COLOR')
    return feature.set('style', {'color': color, 'width': 0})
ecoRegions = ecoRegions.map(add_style_property)

for update in color_updates:
    layer = ecoRegions.filter(ee.Filter.eq('ECO_ID', update['ECO_ID'])).map(
        lambda f: f.set({'COLOR': update['COLOR'], 'style': {'color': update['COLOR'], 'width': 0}})
    )
    ecoRegions = ecoRegions.filter(ee.Filter.neq('ECO_ID', update['ECO_ID'])).merge(layer)

ecoRegions = ecoRegions.style(**{'styleProperty': 'style'})

land_mask = (
    ee.Image(HGFC_ASSET_ID)
    .select("datamask")
    .eq(1)  # 1 = land, 2 = permanent water/ocean, 0 = no data
)

Map.addLayer(ecoRegions.updateMask(land_mask), {}, 'Ecoregions', 0)
Map.addLayer(site_geom, {"color": "white"}, "Test site", 1, 0.5)
Map.addLayer(grid_fc, {"color": "gray"}, "Candidate Cells")
Map.addLayer(matched_grids, {"color": "yellow"}, "Matched Cells")

Map.centerObject(grid_fc)
Map